In [1]:
from prompt import *

print(prompts['GPU'])


    "You are an expert PC hardware analyst and salesman.
Your task is to select the single best GPU from a provided JSON list of available graphics cards based on strict business logic, and then justify your choice to the customer in Russian.

Inputs available in your context:
- `gpu_list`: A list of dicts, where each dict represents a GPU with its specs (normalized_name, price_rub, tdp, length_mm, power_connectors, source_url, test, result).
- `target_resolution`: Numeric (e.g., 1080, 1440, 2160)
- `target_task`: String (the benchmark/task name)

CRITICAL OPERATIONAL RULES:
1. You MUST execute a Python code block to filter `gpu_list` step-by-step according to the Selection Algorithm below. Do not guess the winner; let Python find it.
2. Your final output MUST be a valid JSON object matching the requested schema. Do not prefix it with "Final Answer:" or wrap it in anything other than standard JSON/markdown code blocks.

Selection algorithm — implement exactly in this order:
Step 1: On

In [2]:
def query_database(sql: str) -> str:
    """
    Query the PC components database to get prices, benchmarks and compatibility info.
    Args:
        sql: SQL SELECT query to execute (no semicolons).
    """
    try:
        clean_sql = sql.strip().rstrip(";")
        response = supabase.rpc("run_query", {"sql": clean_sql}).execute()
        return json.dumps(response.data, ensure_ascii=False, indent=2) if response.data else "[]"
    except Exception as e:
        return f"Database Error: {e}"

from dotenv import load_dotenv
from supabase import create_client
import os
load_dotenv()

url = os.getenv("url")
key = os.getenv("key")
supabase = create_client(url, key)

In [3]:
min_p = 38700
max_p = 60000
target_task = 'Relative Performance TechPowerUp'
target_resolution = 1080

In [4]:
from dotenv import load_dotenv
from supabase import create_client
import os
import json
load_dotenv()

url = os.getenv("url")
key = os.getenv("key")
supabase = create_client(url, key)

def query_database(sql: str) -> str:
    """
    Query the PC components database to get prices, benchmarks and compatibility info.
    Args:
        sql: SQL SELECT query to execute (no semicolons).
    """
    try:
        clean_sql = sql.strip().rstrip(";")
        response = supabase.rpc("run_query", {"sql": clean_sql}).execute()
        return json.dumps(response.data, ensure_ascii=False, indent=2) if response.data else "[]"
    except Exception as e:
        return f"Database Error: {e}"

In [5]:
from tools import *
gpu_list = get_gpu(0, 50000, 'Relative Performance TechPowerUp', 1080)

In [6]:
import os
from dotenv import load_dotenv

# ── supabase ──────────────────────────────────────────────────────────────────
_hf_token      = os.getenv("HF_TOKEN")
_supabase_url  = os.getenv("url")
_supabase_key  = os.getenv("key")

from smolagents import CodeAgent, InferenceClientModel, OpenAIModel, tool
import ast
@tool
def query_database(sql: str) -> str:
    """
    Query the PC components database to get prices, benchmarks and compatibility info.
    Args:
        sql: SQL SELECT query to execute (no semicolons).
    """
    try:
        clean_sql = sql.strip().rstrip(";")
        response = supabase.rpc("run_query", {"sql": clean_sql}).execute()
        return json.dumps(response.data, ensure_ascii=False, indent=2) if response.data else "[]"
    except Exception as e:
        return f"Database Error: {e}"

# ── helpers ───────────────────────────────────────────────────────────────────
def get_available_tests() -> list[str]:
    try:
        response = supabase.rpc("run_query", {"sql": "SELECT distinct(test) FROM model_x_test"}).execute()
        return [row["test"] for row in response.data] if response.data else []
    except Exception:
        return ["Relative Performance TechPowerUp"]

def _parse_agent_output(raw) -> dict | str:
    """Пробует привести ответ агента к dict; если не выходит — возвращает как есть."""
    if isinstance(raw, dict):
        return raw
    clean = str(raw).replace("Final answer:", "").strip()
    try:
        return ast.literal_eval(clean)
    except (ValueError, SyntaxError):
        return clean

# ── model / agent factories ───────────────────────────────────────────────────
def make_model(provider: str):
    match provider:
        case "hf" | "huggingface":
            return InferenceClientModel(
                model_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",
                token=_hf_token,
            )
        case "groq":
            return OpenAIModel(
                model_id="llama-3.3-70b-versatile",
                api_base="https://api.groq.com/openai/v1",
                api_key=os.getenv("GROQ_API_KEY"),
            )
        case "google":
            return OpenAIModel(
                model_id="gemini-2.0-flash",
                api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
                api_key=os.getenv("GOOGLE_API_KEY"),
            )
        case _:
            raise ValueError(f"Неизвестный провайдер: {provider}")

def make_agent(model, sys_prompt: str, name: str, max_steps: int = 8) -> CodeAgent:
    return CodeAgent(
        tools=[query_database],
        model=model,
        code_block_tags=("```python", "```"),
        additional_authorized_imports=["json", "re", "pandas"],
        instructions=sys_prompt,
        max_steps=max_steps,
        verbosity_level=0,          # rich-вывод заглушён; логи идут в файл
    )

# бюджетные доли: [min, max] для каждого компонента
RATIO = {
    "GPU":      [0.38, 0.51],
    "CPU_MB":   [0.14, 0.22],
    "RAM":      [0.10, 0.16],
    "PSU":      [0.04, 0.08],
    "STORAGE":  [0.08, 0.16],
    "CASE":     [0.02, 0.06],
    "COOLER":   [0.01, 0.04],
}

if __name__ == "__main__":
    # ── main flow ─────────────────────────────────────────────────────────────────
    available_tests = get_available_tests()
    model_ll = make_model("hf")

C:\Users\ya\Documents\agent_try\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
agent_gpu = make_agent(model_ll, prompts["GPU"], name="GPU")
gpu_input = f"""
    gpu_list: {gpu_list},
    target_resolution: {target_resolution},
    target_task: {target_task},
"""

In [8]:
chosen_gpu = agent_gpu.run(gpu_input)

In [9]:
from pydantic import BaseModel, ValidationError, conint, confloat, field_validator

class GPU(BaseModel):
    normalized_name: str
    price_rub: float = confloat(ge=RATIO["GPU"][0], lt=RATIO["GPU"][1])
    tdp: float = confloat(ge=10, le=800)
    length_mm: float = confloat(ge=120.0, le=400.0)
    power_connectors: int  = conint(ge=6, le=24)
    source_url: str
    explanation_ru: str

    @field_validator('source_url')
    def check_wildberries_domain(cls, v):
        if '.wildberries.ru' not in v:
            raise ValueError("incorrect source_url")
        return v

validation_error = None

try:
    valid_gpu = GPU.model_validate(chosen_gpu).model_dump()
except ValidationError as e:
    validation_error = e.errors()
    chosen_gpu = agent_gpu.run('format: ' + str(validation_error) + 'GPUs: ' + gpu_input)

In [10]:
chosen_gpu

{'normalized_name': 'MSI GeForce RTX 5060 Ti VENTUS 3X OC 16G',
 'price_rub': 47865,
 'tdp': 180,
 'length_mm': 306,
 'power_connectors': 8,
 'source_url': 'https://www.wildberries.ru/catalog/502308707/detail.aspx',
 'explanation_ru': 'Видеокарта MSI GeForce RTX 5060 Ti VENTUS 3X OC 16G показывает один из лучших результатов в выбранном 벤치марке.'}

{'A': 2, 'B': 2}
